# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u

In [ ]:
import matplotlib as mpl
mpl.rcParams['text.usetex'] = False
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 8


In [ ]:
!pip install astroquery
from astroquery.sdss import SDSS

# Loading in Galaxy Sample

In [ ]:
url = "https://raw.githubusercontent.com/cthitch137/Galaxy-Population-Study/refs/heads/main/data/galaxy_sample.csv"
df = pd.read_csv(url)

In [ ]:
df

---
## Pulling emission line data from SDSS for BPT classification

We're looking to pull four emission lines from SDSS: **[OIII], Hβ, [NII], Hα**.

By taking the ratio of the lines (OIII/Hβ and NII/Hα), we can tell what is ionizing the gas in a galaxy. When looking for AGNs, their high energy ionization produces different ratios than those from young hot stars in star forming regions.



In [ ]:
query = f"""
SELECT TOP 3600
  s.bestObjID as objid,
  g.oiii_5007_flux, g.oiii_5007_flux_err,
  g.h_beta_flux, g.h_beta_flux_err,
  g.nii_6584_flux, g.nii_6584_flux_err,
  g.h_alpha_flux, g.h_alpha_flux_err
FROM galSpecLine g
JOIN SpecObj s ON g.specobjid = s.specobjid
WHERE s.z BETWEEN 0.02 AND 0.25
AND s.class = 'GALAXY'
AND s.sciencePrimary = 1
"""

result = SDSS.query_sql(query)
df_lines = result.to_pandas()

df_bpt = df.merge(df_lines, on='objid', how='inner')
df_bpt = df_bpt.drop_duplicates(subset='objid', keep='first')
df_bpt = df_bpt.reset_index(drop=True)
print(f"Galaxies with emission line data: {len(df_bpt)}")


In [ ]:
print(f"Full sample: {len(df)}")
print(f"Galaxies with emission line data: {len(df_bpt)}")
print(f"Fraction of full sample with emission lines: {len(df_bpt)/len(df):.2f}")

In [ ]:
sn_cut = (
    (df_bpt['oiii_5007_flux'] / df_bpt['oiii_5007_flux_err'] > 3) &
    (df_bpt['h_beta_flux'] / df_bpt['h_beta_flux_err'] > 3) &
    (df_bpt['nii_6584_flux'] / df_bpt['nii_6584_flux_err'] > 3) &
    (df_bpt['h_alpha_flux'] / df_bpt['h_alpha_flux_err'] > 3)
)

df_bpt = df_bpt[sn_cut]

In [ ]:
df_bpt['log_nii_ha'] = np.log10(df_bpt['nii_6584_flux'] / df_bpt['h_alpha_flux'])
df_bpt['log_oiii_hb'] = np.log10(df_bpt['oiii_5007_flux'] / df_bpt['h_beta_flux'])

In [ ]:
bpt_range = (
    (df_bpt['log_nii_ha'].between(-2, 1)) &
    (df_bpt['log_oiii_hb'].between(-1.5, 1.5))
)

print(f"After range cuts: {len(df_bpt[bpt_range])}")

df_bpt = df_bpt[bpt_range]

In [ ]:
x = np.linspace(-1.5, 0, 100)

kauf = 0.61 / (x - 0.05) + 1.3  # Kauffmann et al. 2003 - empirical SF/composite boundary
kewl = 0.61 / (x - 0.47) + 1.19 # Kewley et al. 2001 - theoretical maximum starburst line

sf = (df_bpt['log_nii_ha'] < 0.05) & (df_bpt['log_oiii_hb'] < (0.61 / (df_bpt['log_nii_ha'] - 0.05) + 1.3))   # Star forming
agn = (df_bpt['log_nii_ha'] >= 0.47) | (df_bpt['log_oiii_hb'] > (0.61 / (df_bpt['log_nii_ha'] - 0.47) + 1.19)) # AGNs
comp = ~sf & ~agn

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(x, kauf, 'k--', label="Kauffmann+03")
plt.plot(x, kewl, 'k-', label="Kewley+01")

plt.scatter(df_bpt['log_nii_ha'], df_bpt['log_oiii_hb'], c='k', alpha=0.2, label=f"Galaxies: {len(df_bpt)}")
plt.scatter(df_bpt[sf]['log_nii_ha'], df_bpt[sf]['log_oiii_hb'], c='blue', alpha=0.6, label=f"Star forming: {sf.sum()}")
plt.scatter(df_bpt[comp]['log_nii_ha'], df_bpt[comp]['log_oiii_hb'], c='green', alpha=0.6, label=f"Composite: {comp.sum()}")
plt.scatter(df_bpt[agn]['log_nii_ha'], df_bpt[agn]['log_oiii_hb'], c='red', alpha=0.6, label=f"AGN: {agn.sum()}")


#plt.xlim(-2, 1)
plt.ylim(-1.5, 1.5)
plt.xlabel(r'$\log_{10}(\text{[NII]} \, \lambda \, 6584 \text{/Hα})$', fontsize = 14)
plt.ylabel(r'$\log_{10}(\text{[OIII]} \, \lambda \, 5007 \text{/Hβ})$', fontsize = 14)

plt.legend()

In [ ]:
df_bpt['bpt_class'] = 'composite'
df_bpt.loc[sf, 'bpt_class'] = 'star_forming'
df_bpt.loc[agn, 'bpt_class'] = 'agn'

In [ ]:
df = df.merge(df_bpt[['objid', 'bpt_class']], on='objid', how='left')
df['bpt_class'] = df['bpt_class'].fillna('no_emission')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))

axes[0].hist(df[df['bpt_class']=='star_forming']['modelMag_r'], color='red', label='Star Forming')
axes[0].hist(df[df['bpt_class']=='agn']['modelMag_r'], color='blue', alpha=0.6, label='AGN')
axes[0].set_xlabel('modelMag_r', fontsize=14)
axes[0].legend()

axes[1].hist(df[df['bpt_class']=='star_forming']['z'], color='red', label='Star Forming')
axes[1].hist(df[df['bpt_class']=='agn']['z'], color='blue', alpha=0.6, label='AGN')
axes[1].set_xlabel('Redshift', fontsize=14)
axes[1].legend()

# SED Fitting with BAGPIPES
(Bayesian Analysis of Galaxies for Physical Inference and Parameter EStimation)

BAGPIPES has four key ingredients:
1. Filter Curves:            - This tells BAGPIPES the wavelengths observed
2. ```load_data``` function  - Feeds the flux measurements into BAGPIPES
3. ```fit_instructions```    - Defines the model (SFH, dust, redshift)
4. Running the fit           - bagpipes.fit()

## Initial step: Installing BAGPIPES and importing

In [ ]:
!pip install bagpipes
import bagpipes as pipes

## Step 1: Filters
SDSS *ugriz* filter curves need to be downloaded into ```txt``` files. We'll grab them directly from the SVO Filter Profile Service.

In [ ]:
import requests, os

filter_urls = {
    'u': 'https://svo2.cab.inta-csic.es/theory/fps/getdata.php?format=ascii&id=SLOAN/SDSS.u',
    'g': 'https://svo2.cab.inta-csic.es/theory/fps/getdata.php?format=ascii&id=SLOAN/SDSS.g',
    'r': 'https://svo2.cab.inta-csic.es/theory/fps/getdata.php?format=ascii&id=SLOAN/SDSS.r',
    'i': 'https://svo2.cab.inta-csic.es/theory/fps/getdata.php?format=ascii&id=SLOAN/SDSS.i',
    'z': 'https://svo2.cab.inta-csic.es/theory/fps/getdata.php?format=ascii&id=SLOAN/SDSS.z'
}

os.makedirs('filters', exist_ok=True)

for band, url in filter_urls.items():
  r = requests.get(url)
  with open(f'filters/sdss_{band}.txt', 'w') as f:
    f.write(r.text)
  print(f'Downloaded {band}')

In [ ]:
filt_list = [f'filters/sdss_{b}.txt' for b in ['u', 'g', 'r', 'i', 'z']]
filt_list

In [ ]:
test_gal = df[df['bpt_class'] == 'star_forming'].iloc[0]
print(test_gal[['objid', 'z', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']])

## Magnitude to flux conversion
BAGPIPES takes in flux rather than magnitude. The units of flux should also be in micro Janskys (μJy).

In [ ]:
def mag2flux(mag): return 10**((23.9 - mag) / 2.5)
def magerr2fluxerr(mag, magerr): return mag2flux(mag) * magerr * np.log(10) / 2.5

In [ ]:
df_test = df

In [ ]:
bands = ['u', 'g', 'r', 'i', 'z']
for band in bands:
  test_gal[f'Flux_{band}'] = mag2flux(test_gal[f'modelMag_{band}'])
  test_gal[f'FluxErr_{band}'] = magerr2fluxerr(test_gal[f'modelMag_{band}'], test_gal[f'modelMagErr_{band}'])


In [ ]:
Flux_test = test_gal[['Flux_u', 'Flux_g', 'Flux_r', 'Flux_i', 'Flux_z']]
FluxErr_test = test_gal[['FluxErr_u', 'FluxErr_g', 'FluxErr_r', 'FluxErr_i', 'FluxErr_z']]

In [ ]:
phot_test = np.c_[Flux_test, FluxErr_test]

## ```load_data``` function
BAGPIEPS wants the photometry and spectroscopy data in a ```(5,2)``` numpy array of ```[[Flux, Flux Error], ...]```. When inputting this data, we'll want the input of the function to be a galaxy id (```objid```) as it runs over each galaxy individually.

In [ ]:
def load_data(ID):
  gal_row = df[df['objid'] == int(ID)].iloc[0]

  bands = ['u', 'g', 'r', 'i', 'z']

  Flux = [mag2flux(gal_row[f'modelMag_{band}']) for band in bands]
  FluxErr = [magerr2fluxerr(gal_row[f'modelMag_{band}'], gal_row[f'modelMagErr_{band}']) for band in bands]

  phot = np.column_stack([Flux, FluxErr])

  return phot

In [ ]:
load_data(1237648720142336185)

## Fit Instructions
Here we define the model: exponential star formation history, dust, and redshift.

In [ ]:
test_gal = df[df['bpt_class'] == 'agn'].iloc[28]
print(test_gal[['objid', 'bpt_class', 'z', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']])

In [ ]:
test_gal = df[df['objid'] == 1237648720142073942].iloc[0]
print(test_gal[['objid', 'bpt_class', 'z', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']])

In [ ]:
# import inspect
# from bagpipes.models import agn_model
# print(inspect.getsource(agn_model.agn.update))

In [ ]:
exp= {}
exp["tau"] = (0.1, 15)
exp["massformed"] = (1, 13)
exp["age"] = (0.1, 13)
exp["metallicity"] = (0.1, 2.5)

dust = {}
dust["type"] = "Calzetti"
dust["Av"] = (0, 4)

agn = {}
agn["alphalam"] = -1.5
agn["betalam"] = -0.5
#agn["f_agn"] = (0, 3)
agn["f5100A"] = (1e-17, 1e-15)
agn["sigma"] = 1000
agn["hanorm"] = 0

fit_instr = {}
fit_instr["exponential"] = exp
fit_instr["dust"] = dust
fit_instr["redshift"] = test_gal['z']
fit_instr["agn"] = agn

In [ ]:
# # UNCOMMENT FOR SINGLE TEST GALAXY
# galaxy = pipes.galaxy(test_gal['objid'], load_data, filt_list = filt_list, spectrum_exists=False)

# fit = pipes.fit(galaxy, fit_instr, run = "Test_fit")
# fit.fit(verbose=False)

In [ ]:
# # UNCOMMENT FOR TEST GALAXY PLOT
# fig = fit.plot_spectrum_posterior(save=True, show=True)
# fig = fit.plot_sfh_posterior(save=True, show=True)
# fig = fit.plot_corner(save=True, show=True)

In [ ]:
agn_sample_ids = (
    df[df['bpt_class'] == 'agn']['objid'].iloc[:].tolist()
)

sf_sample_ids = (
    df[df['bpt_class'] == 'star_forming']['objid'].iloc[:46].tolist()
)

In [ ]:
import signal
import time

def timeout(signum, frame):
  raise TimeoutError("Fit timed out after 15 minutes")


def fit_model(objid_list, run_name="fit_model_run", fit_agn=False):
  start_time = time.time()
  signal.signal(signal.SIGALRM, timeout)

  exp= {}
  exp["tau"] = (0.1, 15)
  exp["massformed"] = (1, 13)
  exp["age"] = (0.1, 13)
  exp["metallicity"] = (0.1, 2.5)

  dust = {}
  dust["type"] = "Calzetti"
  dust["Av"] = (0, 4)

  fit_instr = {}
  fit_instr["exponential"] = exp
  fit_instr["dust"] = dust

  if fit_agn:
    agn = {}
    agn["alphalam"] = -1.5
    agn["betalam"] = -0.5
    agn["f5100A"] = (1e-17, 1e-15)
    agn["sigma"] = 1000
    agn["hanorm"] = 0
    fit_instr["agn"] = agn

  results = []
  if isinstance(objid_list, (int, np.integer)):
    objid_list = [objid_list]
  track = 0
  for id in objid_list:
    print(f"Fitting {id}...")
    try:
      signal.alarm(720)

      fit_instr["redshift"] = df[df['objid']==id].iloc[0]['z']

      galaxy = pipes.galaxy(id, load_data, filt_list = filt_list, spectrum_exists=False)
      fit = pipes.fit(galaxy, fit_instr, run = run_name)
      fit.fit(verbose=False)

      signal.alarm(0)

      s = fit.posterior.samples
      results.append({
          'objid': id,
          'bpt_class': df[df['objid'] == id].iloc[0]['bpt_class'],
          'Av_50': np.percentile(s['dust:Av'], 50),
          'Av_16': np.percentile(s['dust:Av'], 16),
          'Av_84': np.percentile(s['dust:Av'], 84),
          'age_50': np.percentile(s['exponential:age'], 50),
          'age_16': np.percentile(s['exponential:age'], 16),
          'age_84': np.percentile(s['exponential:age'], 84),
          'mass_50': np.percentile(s['exponential:massformed'], 50),
          'mass_16': np.percentile(s['exponential:massformed'], 16),
          'mass_84': np.percentile(s['exponential:massformed'], 84),
          'metallicity_50': np.percentile(s['exponential:metallicity'], 50),
          'metallicity_16': np.percentile(s['exponential:metallicity'], 16),
          'metallicity_84': np.percentile(s['exponential:metallicity'], 84),
          'tau_50': np.percentile(s['exponential:tau'], 50),
          'tau_16': np.percentile(s['exponential:tau'], 16),
          'tau_84': np.percentile(s['exponential:tau'], 84),
      })
      if fit_agn:
        results['f5100A_50'] = np.percentile(s['agn:f5100A'], 50)
        results['f5100A_16'] = np.percentile(s['agn:f5100A'], 16)
        results['f5100A_84'] = np.percentile(s['agn:f5100A'], 84)


      df_results = pd.DataFrame(results)
      df_results.to_csv('bagpipes_results.csv', index=False)
      fig = fit.plot_spectrum_posterior(save=True)
      fig = fit.plot_sfh_posterior(save=True)
      fig = fit.plot_corner(save=True)

      plt.close('all')
      track+=1
      print(f"Targets remaining {len(objid_list)-track} / {len(objid_list)}")
      print("_"*29)
      print()
    except Exception as e:
      signal.alarm(0)
      print(f"Skipped {id}: {e}")
      track+=1
      plt.close('all')


  print(f"Finished fitting models after {time.time() - start_time:.2f} seconds.")
  print()

  return df_results

In [ ]:
# # AGN SAMPLE
df_results = fit_model(agn_sample_ids, "AGN_sample", fit_agn=True)

# STAR-FORMING SAMPLE
df_results = fit_model(sf_sample_ids, "SF_sample")

import shutil
shutil.make_archive('pipes', 'zip', 'pipes')

# Data Analysis

In [ ]:
df_results_agn = pd.read_csv('/content/bagpipes_results_agn.csv')
df_results_sf  = pd.read_csv('/content/bagpipes_results_sf.csv')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].hist(df_results_agn['mass_50'], bins=15, label="AGN")
axes[1].hist(df_results_agn['age_50'], bins=15, label="AGN")
axes[2].hist(df_results_agn['Av_50'], bins=15, label="AGN")

axes[0].hist(df_results_sf['mass_50'], bins=15, alpha=0.6, label="Star forming")
axes[1].hist(df_results_sf['age_50'], bins=15, alpha=0.6, label="Star forming")
axes[2].hist(df_results_sf['Av_50'], bins=15, alpha=0.6, label="Star forming")

axes[0].set_xlabel('mass_50', fontsize=14)
axes[0].legend()
axes[1].set_xlabel('age_50', fontsize=14)
axes[1].legend()
axes[2].set_xlabel('Av_50', fontsize=14)
axes[2].legend()



In [ ]:
print("AGN")
print(" Mean Mass:")
print(f" Lower Bound (16): {np.mean(df_results_agn['mass_16']):.2f} \t Mid (50): {np.mean(df_results_agn['mass_50']):.2f} \t Upper Bound (84): {np.mean(df_results_agn['mass_84']):.2f}")
print('-'*90)

print(" Mean Age:")
print(f" Lower Bound (16): {np.mean(df_results_agn['age_16']):.2f} \t Mid (50): {np.mean(df_results_agn['age_50']):.2f} \t Upper Bound (84): {np.mean(df_results_agn['age_84']):.2f}")
print('-'*90)

print(" Mean Av:")
print(f" Lower Bound (16): {np.mean(df_results_agn['Av_16']):.2f} \t Mid (50): {np.mean(df_results_agn['Av_50']):.2f} \t Upper Bound (84): {np.mean(df_results_agn['Av_84']):.2f}")
print('-'*90)

print(" Mean Tau:")
print(f" Lower Bound (16): {np.mean(df_results_agn['tau_16']):.2f} \t Mid (50): {np.mean(df_results_agn['tau_50']):.2f} \t Upper Bound (84): {np.mean(df_results_agn['tau_84']):.2f}")
print('-'*90)

print('_'*90)

print("STAR FORMING")
print(" Mean Mass:")
print(f" Lower Bound (16): {np.mean(df_results_sf['mass_16']):.2f} \t Mid (50): {np.mean(df_results_sf['mass_50']):.2f} \t Upper Bound (84): {np.mean(df_results_sf['mass_84']):.2f}")
print('-'*90)

print(" Mean Age:")
print(f" Lower Bound (16): {np.mean(df_results_sf['age_16']):.2f} \t Mid (50): {np.mean(df_results_sf['age_50']):.2f} \t Upper Bound (84): {np.mean(df_results_sf['age_84']):.2f}")
print('-'*90)

print(" Mean Av:")
print(f" Lower Bound (16): {np.mean(df_results_sf['Av_16']):.2f} \t Mid (50): {np.mean(df_results_sf['Av_50']):.2f} \t Upper Bound (84): {np.mean(df_results_sf['Av_84']):.2f}")
print('-'*90)

print(" Mean Tau:")
print(f" Lower Bound (16): {np.mean(df_results_sf['tau_16']):.2f} \t Mid (50): {np.mean(df_results_sf['tau_50']):.2f} \t Upper Bound (84): {np.mean(df_results_sf['tau_84']):.2f}")
print('-'*90)

In [ ]:
from scipy import stats

In [ ]:
for param in ["mass_50", "age_50", "Av_50"]:
  stat, p = stats.ks_2samp(df_results_agn[param], df_results_sf[param])
  print(f" {param}: \t KS stat = {stat:.3f} \t p-value = {p:.4f}")